## Transform Circuits Data
1. Read bronze circuits file.
1. Keep only the columns required for analytics (DRop URL columns)
1. Standardise column names using snake_case.
1. Rename columns to make them meaning ful (lat-> latitude, etc)
1. Filter out rows where circuit_id is null.
1. Remove duplicate records.
1. Transform values of columns circuit_name and locality to Title Case
1. Write the transformed data to silver circuits table

In [0]:
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/03.silver-helpers

In [0]:
bronze_table= f"{catalog_name}.{bronze_schema}.circuits"
silver_table= f"{catalog_name}.{silver_schema}.circuits"

## Read bronze circuits file

In [0]:
#circuits_df= spark.read.option('versionAsOf =', 0).table(bronze_table)

In [0]:
circuits_df=( spark.table(bronze_table)
             .filter((F.col("batch_id")== v_batch_id))
             
)

## Keep only the columns required for analytics (drop URL)

In [0]:
# circuits_select_df =circuits_df.select(
#     "circuitId",
#     "circuitName",
#     "lat",
#     "lng",
#     "locality",
#     "country",
#     "Ingestion_timestamp",
#     "source_file"
# ) 

In [0]:
circuits_select_df =circuits_df.select(
    F.col("circuitId"),
    F.col("circuitName"),
    F.col("lat"),
    F.col("lng"),
    F.col("locality"),
    F.col("country"),
    F.col("Ingestion_timestamp"),
    F.col("source_file"),
    F.col("batch_id")
)

### Step 3 & 4 - Standardise Columns Names
- Standardise column names using snake case (circuitId -> circuit_id, circuitName->circuit_name)
- Rename columns to make them more meaning full( lat - latitude, lng - longitute)

In [0]:
# circuits_renamed_df = (
#     circuits_select_df
#     .withColumnRenamed("circuitId", "circuit_id")
#     .withColumnRenamed("circuitName", "circuit_name")
#     .withColumnRenamed("lat","latitude")
#     .withColumnRenamed("lng","longitude")
#     )

In [0]:
circuits_renamed_df = (
    circuits_select_df
    .withColumnsRenamed({
        "circuitId": "circuit_id",
        "circuitName": "circuit_name",
        "lat":"latitude",
        "lng":"longitude"
        })
    )

## Filtering out rows where circuit_id is null (business validation)

In [0]:
# circuits_valid_df = circuits_renamed_df.filter(
#     "circuit_id IS NOT NULL"
# )

In [0]:
circuits_valid_df = circuits_renamed_df.filter(
    F.col("circuit_id").isNotNull()
)

In [0]:
display(circuits_valid_df)

#### Remove duplicate records

In [0]:
# circuits_distinct_df= circuits_valid_df.distinct()

In [0]:
circuits_distinct_df= circuits_valid_df.dropDuplicates(["circuit_id"])

In [0]:
display(circuits_distinct_df)

#### Transform values of columns circuits_name and locality to Title Case

In [0]:
circuits_final_df= (
circuits_distinct_df
    .withColumn('circuit_name', F.initcap(F.col("circuit_name")))
    .withColumn('locality', F.initcap(F.col("locality")))
)

In [0]:
display(circuits_final_df)

In [0]:
# circuits_final_df = (
#     circuits_final_df
#     .withColumn("created_timestamp", F.current_timestamp())
#     .withColumn("updated_timestamp", F.current_timestamp())
# )

In [0]:
# if not spark.catalog.tableExists(silver_table):
#     (
#         circuits_final_df
#         .write
#         .format("delta")
#         .mode("overwrite")
#         .saveAsTable(silver_table)
#     )
# else:
#         from delta.tables import DeltaTable
#         delta_table = DeltaTable.forName(spark, silver_table)
#         (
#             delta_table.alias("t")
#             .merge (
#                 circuits_final_df.alias("s"),
#                 "t.circuit_id = s.circuit_id"
#             )
#             .whenMatchedUpdate(
#                 condition="s.batch_id >= t.batch_id",
#                 set = {
#                     "circuit_name": "s.circuit_name",
#                     "latitude": "s.latitude",
#                     "longitude": "s.longitude",
#                     "locality": "s.locality",
#                     "country": "s.country",
#                     "Ingestion_timestamp": "s.ingestion_timestamp",
#                     "source_file": "s.source_file",
#                     "batch_id": "s.batch_id",
#                     "updated_timestamp": "s.updated_timestamp"
#                 }
#             )
#             .whenNotMatchedInsertAll()
#             .execute
#         )

In [0]:
write_to_silver(
    input_df= circuits_final_df,
    target_table= silver_table,
    merge_condition= "t.circuit_id= s.circuit_id",
    columns_to_update=[
        "circuit_name",
        "latitude",
        "longitude",
        "locality",
        "country",
        "ingestion_timestamp",
        "source_file",
        "batch_id"
    ]
)

In [0]:
display(spark.table(silver_table))